In [ ]:
# Clonar el repositorio con todo el código y módulos necesarios
# Primero eliminar si existe para asegurar versión actualizada
import sys
%cd /kaggle/working
!rm -rf cells-finder-unsupervised
!git clone https://github.com/martingra/cells-finder-unsupervised.git
%cd cells-finder-unsupervised

# Limpiar módulos importados previamente para forzar recarga
modules_to_remove = [key for key in sys.modules.keys() if key.startswith('utils')]
for mod in modules_to_remove:
    del sys.modules[mod]
print(f"✅ Cache limpiado ({len(modules_to_remove)} módulos removidos)")

# Clustering y Segmentación de Núcleos con BiomedCLIP

Pipeline automático para análisis de imágenes citológicas.

**Instrucciones:**
1. Asegúrate de tener el dataset "cric-dataset" agregado (Settings → Add Dataset)
2. Ejecuta cada celda en orden (Shift+Enter)
3. Primera ejecución puede tardar 5-10 minutos (instalación de dependencias)

In [ ]:
# CELDA 1: SETUP INICIAL (ejecutar primero)
# Instala dependencias y configura todo

import os
import sys
import subprocess

print("⏳ Configurando entorno...\n")

# Instalar dependencias
packages = [
    'open-clip-torch==2.23.0',
    'transformers==4.35.2',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    print(f"✅ {pkg}")

print("\n✅ Setup completado")

In [ ]:
# CELDA 2: IMPORTAR MÓDULOS

import numpy as np
import matplotlib.pyplot as plt
from open_clip import create_model_from_pretrained, get_tokenizer

print("Cargando BiomedCLIP modelo...")
model, preprocess = create_model_from_pretrained(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
)
print("✅ Modelo cargado")

# Importar utilidades
from utils.image_processing import load_image_and_boxes_from_json_cropped
from utils.embeddings import get_all_patch_embeddings_from_image
from utils.multi_step_clustering import (
    run_block_clustering_on_embeddings,
    refinar_cluster_con_kmeans,
    decidir_fondo_vs_tejido,
    decidir_nucleos_vs_citoplasma
)
from utils.evaluation import (
    limpiar_patches_por_componentes_mask,
    agrupar_patches_en_grupos,
    evaluar_grupos_vs_boxes_plus
)
from utils.visualization import visualizar_clusters_basicos, visualizar_limpieza_patches

print("✅ Módulos importados")

In [ ]:
# CELDA 3: PROCESAR UNA IMAGEN (TUTORIAL)
# Cambiar IMAGE_INDEX para procesar otras imágenes

IMAGE_INDEX = 150  # ← CAMBIAR AQUÍ (150-210 disponibles)

JSON_PATH = '/kaggle/input/cric-dataset/classifications.json'
BASE_PATH = '/kaggle/input/cric-dataset'

print(f"Cargando imagen {IMAGE_INDEX}...")
img, boxes, fname = load_image_and_boxes_from_json_cropped(
    JSON_PATH, BASE_PATH, index=IMAGE_INDEX,
    boxes_size=60, block_size=224
)
H, W = img.shape[:2]
print(f"✅ {fname} ({W}x{H})")
print(f"   Ground truth boxes: {len(boxes)}")

In [ ]:
# PASO 1: Fondo vs Tejido

print("Extrayendo embeddings...")
patch_data = get_all_patch_embeddings_from_image(
    img, model, preprocess, tile_size=224
)
print(f"✅ {len(patch_data)} patches extraídos\n")

print("PASO 1: Fondo vs Tejido")
clustered_k2 = run_block_clustering_on_embeddings(patch_data, n_clusters=2)
fondo_id, tejido_id, _ = decidir_fondo_vs_tejido(img, clustered_k2)

visualizar_clusters_basicos(img, clustered_k2, boxes=boxes, title="Paso 1: Fondo vs Tejido")

In [ ]:
# PASO 2: Núcleos vs Citoplasma (sobre tejido)

print("PASO 2: Núcleos vs Citoplasma (sobre tejido)")
paso_2 = refinar_cluster_con_kmeans(
    clustered_k2, cluster_id=tejido_id, nuevo_k=2,
    cluster_field='cluster', new_field='subcluster'
)
nucleos_id, cyto_id, _ = decidir_nucleos_vs_citoplasma(img, paso_2)

visualizar_clusters_basicos(img, paso_2, boxes=boxes, title="Paso 2: Núcleos vs Citoplasma")

In [ ]:
# PASO 3: Núcleo en citoplasma vs Suelto (sobre núcleos)

print("PASO 3: Núcleo en citoplasma vs Suelto (sobre núcleos)")
paso_3 = refinar_cluster_con_kmeans(
    paso_2, cluster_id=nucleos_id, nuevo_k=2,
    cluster_field='subcluster', new_field='subsubcluster'
)
nucleo_en_cito_id, _, _ = decidir_nucleos_vs_citoplasma(img, paso_3, cluster_field='subsubcluster')

visualizar_clusters_basicos(img, paso_3, boxes=boxes, title="Paso 3: Núcleo en Citoplasma vs Suelto")

In [ ]:
# PASO 4: Refinamiento final

print("PASO 4: Refinamiento final")
paso_4 = refinar_cluster_con_kmeans(
    paso_3, cluster_id=nucleo_en_cito_id, nuevo_k=2,
    cluster_field='subsubcluster', new_field='subsubsubcluster'
)
nucleo_objetivo_id, _, _ = decidir_nucleos_vs_citoplasma(img, paso_4, cluster_field='subsubsubcluster')

visualizar_clusters_basicos(img, paso_4, boxes=boxes, title="Paso 4: Refinamiento Final")

In [ ]:
# LIMPIEZA: Eliminar componentes pequeñas

salida = [p for p in paso_4 if p.get('subsubsubcluster') == nucleo_objetivo_id]
print(f"Patches objetivo (pre-limpieza): {len(salida)}")

kept, removed, _ = limpiar_patches_por_componentes_mask(
    img, salida, min_patches=3, dilate_px=32
)
print(f"Después limpieza: {len(kept)} (removidos: {len(removed)})\n")

visualizar_limpieza_patches(img, kept, removed, boxes=boxes)

In [ ]:
# AGRUPACIÓN Y EVALUACIÓN

grupos, _ = agrupar_patches_en_grupos(img, kept, min_patches_por_grupo=3)
print(f"Grupos detectados: {len(grupos)}\n")

metrics = evaluar_grupos_vs_boxes_plus(img, grupos, boxes, match_mode='cover_gt')

print("📊 MÉTRICAS FINALES:")
print(f"  TP (grupos correctos): {metrics['groups_TP']}")
print(f"  FP (grupos incorrectos): {metrics['groups_FP']}")
print(f"  GT cubiertos: {metrics['gt_hit']}/{metrics['gt_total']}")
print(f"  Precisión: {metrics['group_precision']:.3f}")
print(f"  Recall: {metrics['gt_recall_coverage']:.3f}")
print(f"  F1 Score: {metrics['f1_coverage']:.3f}")

## 🎉 Listo!

Puedes:
1. Cambiar `IMAGE_INDEX` en la Celda 3 para procesar otras imágenes
2. Ejecutar nuevamente para ver otros resultados
3. Ver KAGGLE_README.md para opciones avanzadas (procesamiento batch, descarga de resultados, etc.)